# Loading Libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests
from pathlib import Path

# Loading Datsets

In [3]:
RAW_DATA_DIR = Path("__file__").resolve().parent.parent / "data" / "raw"

In [4]:
declarations_df = pd.read_csv(f'{RAW_DATA_DIR}/declarations.csv')
disaster_summaries_df = pd.read_csv(f'{RAW_DATA_DIR}/disaster_summaries.csv')
public_assistance_df = pd.read_csv(f'{RAW_DATA_DIR}/public_assistance.csv')

# Exploratory Data Analysis

## Declaration Summary

In [5]:
declarations_df.shape

(69899, 13)

In [6]:
declarations_df.isna().sum()

disasterNumber         0
state                  0
declarationDate        0
incidentType           0
incidentBeginDate      0
incidentEndDate      535
declarationType        0
ihProgramDeclared      0
iaProgramDeclared      0
paProgramDeclared      0
hmProgramDeclared      0
designatedArea         0
fyDeclared             0
dtype: int64

In [7]:
key_cols = ['disasterNumber', 'declarationDate', 'incidentType', 'designatedArea', 'declarationType', 'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared']
# 1. Which columns vary within duplicate groups?
print("=== Columns that vary within key groups ===")
varying_cols = {}
for col in declarations_df.columns:
    if col in key_cols:
        continue
    n_unique = declarations_df.groupby(key_cols)[col].nunique()
    has_variation = (n_unique > 1).sum()
    if has_variation > 0:
        varying_cols[col] = has_variation

for col, count in sorted(varying_cols.items(), key=lambda x: -x[1]):
    print(f"  {col}: varies in {count} disaster groups")

# 2. How much does designatedArea fan out per disaster?
print("\n=== designatedArea count per disaster ===")
print(declarations_df.groupby('disasterNumber')['designatedArea'].nunique().describe())

# 3. Does incidentType actually vary within a disaster?
print("\n=== incidentType variation within disasterNumber ===")
incident_variation = declarations_df.groupby('disasterNumber')['incidentType'].nunique()
print(incident_variation.value_counts())
print(f"Disasters with >1 incidentType: {(incident_variation > 1).sum()}")

# 4. Sample a high-repeat disaster to eyeball the structure
most_repeated = declarations_df['disasterNumber'].value_counts().idxmax()
print(f"\n=== Sample of most repeated disasterNumber: {most_repeated} ===")
print(declarations_df[declarations_df['disasterNumber'] == most_repeated].to_string())

=== Columns that vary within key groups ===

=== designatedArea count per disaster ===
count    5179.000000
mean       13.491215
std        25.026669
min         1.000000
25%         1.000000
50%         3.000000
75%        14.000000
max       438.000000
Name: designatedArea, dtype: float64

=== incidentType variation within disasterNumber ===
incidentType
1    5179
Name: count, dtype: int64
Disasters with >1 incidentType: 0

=== Sample of most repeated disasterNumber: 4522 ===
      disasterNumber state           declarationDate incidentType         incidentBeginDate           incidentEndDate declarationType  ihProgramDeclared  iaProgramDeclared  paProgramDeclared  hmProgramDeclared                                   designatedArea  fyDeclared
4866            4522    ME  2020-04-04T00:00:00.000Z   Biological  2020-01-20T00:00:00.000Z  2023-05-11T00:00:00.000Z              DR                  1                  0                  0                  0               Indian Township Indian

In [8]:
key_cols = ['disasterNumber', 'declarationDate', 'incidentType', 'declarationType', 'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared']
declarations_df = declarations_df.sort_values(by=key_cols + ['incidentEndDate'], na_position='last')
declarations_df['incidentEndDate'] = declarations_df.groupby(key_cols)['incidentEndDate'].transform('first')

In [9]:
key_cols = ['disasterNumber', 'declarationDate', 'incidentType', 
            'declarationType', 'state', 'fyDeclared', 
            'incidentBeginDate']

declarations_df_collapsed = declarations_df.groupby(key_cols, as_index=False).agg(
    
    # Core geographic feature — count of designated areas

    incidentEndDate=('incidentEndDate', 'first'),
    designatedArea_count=('designatedArea', 'nunique'),
    
    # Program flags — take max (1 if any row has it activated)
    ihProgramDeclared=('ihProgramDeclared', 'max'),
    iaProgramDeclared=('iaProgramDeclared', 'max'),
    paProgramDeclared=('paProgramDeclared', 'max'),
    hmProgramDeclared=('hmProgramDeclared', 'max'),
    
    # Number of programs activated (useful composite signal)
    
)

declarations_df_collapsed

print(declarations_df_collapsed.shape)
print(declarations_df_collapsed['designatedArea_count'].describe())

(5179, 13)
count    5179.000000
mean       13.491215
std        25.026669
min         1.000000
25%         1.000000
50%         3.000000
75%        14.000000
max       438.000000
Name: designatedArea_count, dtype: float64


In [10]:
declarations_df_collapsed.duplicated(subset=key_cols).sum()

np.int64(0)

In [11]:
declarations_df_collapsed.isna().sum()

disasterNumber            0
declarationDate           0
incidentType              0
declarationType           0
state                     0
fyDeclared                0
incidentBeginDate         0
incidentEndDate         307
designatedArea_count      0
ihProgramDeclared         0
iaProgramDeclared         0
paProgramDeclared         0
hmProgramDeclared         0
dtype: int64

In [12]:
declarations_df_collapsed['incident_duration_days'] = (
    pd.to_datetime(declarations_df_collapsed['incidentEndDate']) - 
    pd.to_datetime(declarations_df_collapsed['incidentBeginDate'])
).dt.days

In [13]:
declarations_df_collapsed['is_ongoing'] = declarations_df_collapsed['incidentEndDate'].isna().astype(int)

In [14]:
declarations_df_collapsed['incident_duration_days'] = declarations_df_collapsed.groupby('incidentType')['incident_duration_days'].transform(
    lambda x: x.fillna(x.median())
)

In [15]:
declarations_df_collapsed.isna().sum()

disasterNumber              0
declarationDate             0
incidentType                0
declarationType             0
state                       0
fyDeclared                  0
incidentBeginDate           0
incidentEndDate           307
designatedArea_count        0
ihProgramDeclared           0
iaProgramDeclared           0
paProgramDeclared           0
hmProgramDeclared           0
incident_duration_days      0
is_ongoing                  0
dtype: int64

In [16]:
# Cap negative durations at 0
declarations_df_collapsed['incident_duration_days'] = declarations_df_collapsed['incident_duration_days'].clip(lower=0)

In [17]:
print(declarations_df_collapsed.groupby(['incidentType', 'is_ongoing'])['incident_duration_days'].describe())

                                 count         mean          std     min  \
incidentType        is_ongoing                                             
Biological          0            167.0  1207.000000     0.000000  1207.0   
Chemical            0              1.0    11.000000          NaN    11.0   
Coastal Storm       0             31.0    11.967742    15.940481     0.0   
Dam/Levee Break     0              6.0    11.166667    14.838014     0.0   
Drought             0             46.0    13.891304    48.727920     0.0   
Earthquake          0             36.0    35.833333    64.665071     0.0   
Fire                0           1434.0    13.395397    42.147887     0.0   
                    1            302.0     6.000000     0.000000     6.0   
Fishing Losses      0              6.0    91.500000   100.233228     0.0   
Flood               0            916.0    11.852620    24.979491     0.0   
                    1              2.0     0.000000     0.000000     0.0   
Freezing    

In [18]:
# Quick check
print(declarations_df_collapsed[declarations_df_collapsed['incident_duration_days'] < 0][['incidentType', 'incidentBeginDate', 'incidentEndDate']].head(10))

Empty DataFrame
Columns: [incidentType, incidentBeginDate, incidentEndDate]
Index: []


## Public Assistance && Disaster Summary EDA

In [20]:
public_assistance_df.shape

(811609, 9)

In [21]:
public_assistance_df.isna().sum()

disasterNumber           0
stateAbbreviation        0
incidentType             0
projectAmount            0
federalShareObligated    0
totalObligated           0
projectSize              0
damageCategoryCode       0
applicantId              5
dtype: int64

In [22]:
disaster_summaries_df.shape

(3940, 10)

In [23]:
disaster_summaries_df.isna().sum()

disasterNumber                   0
totalAmountIhpApproved        3334
totalAmountHaApproved         3390
totalAmountOnaApproved        3336
totalObligatedAmountPa         955
totalObligatedAmountCatAb     1215
totalObligatedAmountCatC2g    2387
totalObligatedAmountHmgp      1101
paLoadDate                     955
iaLoadDate                    3334
dtype: int64

In [24]:
public_assistance_df['damageCategoryCode']

0         B
1         E
2         B
3         C
4         B
         ..
811604    G
811605    E
811606    F
811607    E
811608    E
Name: damageCategoryCode, Length: 811609, dtype: object

In [25]:
public_assistance_df['disasterNumber'].nunique()

1765

In [26]:
public_assistance_agg_df = public_assistance_df.groupby('disasterNumber').agg(
    
    # Cost components (features, not target)
    total_obligated=('totalObligated', 'sum'),
    federal_obligated=('federalShareObligated', 'sum'),
    avg_project_cost=('projectAmount', 'mean'),
    
    # Scale features
    applicant_count=('applicantId', 'nunique'),
    project_count=('projectAmount', 'count'),
    
    # Project size mix
    large_project_count=('projectSize', lambda x: (x == 'Large').sum()),
    small_project_count=('projectSize', lambda x: (x == 'Small').sum()),
    
    # Damage category flags
    has_debris=('damageCategoryCode', lambda x: (x == 'A').any()),
    has_emergency=('damageCategoryCode', lambda x: (x == 'B').any()),
    has_roads=('damageCategoryCode', lambda x: (x == 'C').any()),
    has_water=('damageCategoryCode', lambda x: (x == 'D').any()),
    has_buildings=('damageCategoryCode', lambda x: (x == 'E').any()),
    has_utilities=('damageCategoryCode', lambda x: (x == 'F').any()),
    has_parks=('damageCategoryCode', lambda x: (x == 'G').any()),

).reset_index()

In [27]:
bool_cols = [c for c in public_assistance_agg_df.columns if c.startswith('pa_has_')]
public_assistance_agg_df[bool_cols] = public_assistance_agg_df[bool_cols].astype(int)

In [28]:
print(f"PA collapsed: {public_assistance_agg_df.shape}")

PA collapsed: (1765, 15)


In [29]:
cost_cols = [
    'totalAmountIhpApproved', 'totalAmountHaApproved',
    'totalAmountOnaApproved', 'totalObligatedAmountPa',
    'totalObligatedAmountCatAb', 'totalObligatedAmountCatC2g',
    'totalObligatedAmountHmgp'
]
disaster_summaries_df[cost_cols] = disaster_summaries_df[cost_cols].fillna(0)

In [30]:
disaster_summaries_df['total_disaster_cost'] = (
    disaster_summaries_df['totalAmountIhpApproved'] +
    disaster_summaries_df['totalObligatedAmountPa'] +
    disaster_summaries_df['totalObligatedAmountHmgp']
)

In [31]:
disaster_summaries_df.isna().sum()

disasterNumber                   0
totalAmountIhpApproved           0
totalAmountHaApproved            0
totalAmountOnaApproved           0
totalObligatedAmountPa           0
totalObligatedAmountCatAb        0
totalObligatedAmountCatC2g       0
totalObligatedAmountHmgp         0
paLoadDate                     955
iaLoadDate                    3334
total_disaster_cost              0
dtype: int64

## Investigating Row Discrepancies

In [32]:
decl_ids = set(declarations_df_collapsed['disasterNumber'])
summ_ids = set(disaster_summaries_df['disasterNumber'])

# Question 1: Are they in declarations but not summaries, or vice versa?
in_decl_not_summ = decl_ids - summ_ids
in_summ_not_decl = summ_ids - decl_ids

print(f"In declarations but not summaries: {len(in_decl_not_summ)}")
print(f"In summaries but not declarations: {len(in_summ_not_decl)}")

In declarations but not summaries: 1239
In summaries but not declarations: 0


In [33]:
missing_in_summ = declarations_df_collapsed[declarations_df_collapsed['disasterNumber'].isin(in_decl_not_summ)]
print(f"Disaster Dates missing in summaries:\n{missing_in_summ['incidentBeginDate'].sort_index().value_counts()}")

Disaster Dates missing in summaries:
incidentBeginDate
1996-01-06T00:00:00.000Z    9
1996-01-19T00:00:00.000Z    7
1955-08-20T00:00:00.000Z    6
1985-09-27T00:00:00.000Z    6
1997-02-28T00:00:00.000Z    6
                           ..
1975-07-23T00:00:00.000Z    1
1975-07-25T00:00:00.000Z    1
1975-08-22T00:00:00.000Z    1
1975-09-11T00:00:00.000Z    1
2020-01-20T00:00:00.000Z    1
Name: count, Length: 979, dtype: int64


In [38]:
missing_in_summ['year'] = pd.to_datetime(missing_in_summ['incidentBeginDate']).dt.year
print(missing_in_summ['year'].value_counts().sort_index())

year
1953    13
1954    17
1955    18
1956    16
1957    16
1958     7
1959     7
1960    12
1961    12
1962    22
1963    20
1964    25
1965    25
1966    11
1967    10
1968    18
1969    29
1970    17
1971    17
1972    48
1973    46
1974    46
1975    38
1976    28
1977    22
1978    25
1979    41
1980    23
1981    16
1982    23
1983    24
1984    32
1985    28
1986    27
1987    24
1988    10
1989    33
1990    44
1991    34
1992    46
1993    31
1994    36
1995    36
1996    80
1997    37
1998    42
2011     1
2018     5
2020     1
Name: count, dtype: int64


/var/folders/3g/lzxm0r5j6d9d_qgwb163853r0000gn/T/ipykernel_51403/4239623330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_in_summ['year'] = pd.to_datetime(missing_in_summ['incidentBeginDate']).dt.year


In [37]:
# Confirm the spread
print(f"Earliest missing: {pd.to_datetime(missing_in_summ['incidentBeginDate']).min()}")
print(f"Latest missing: {pd.to_datetime(missing_in_summ['incidentBeginDate']).max()}")

# Are they concentrated in specific types?
print("\nMissing by incident type:")
print(missing_in_summ   ['incidentType'].value_counts())

# Are they low cost disasters? Check if they have PA records
missing_with_pa = missing_in_summ[missing_in_summ['disasterNumber'].isin(public_assistance_agg_df['disasterNumber'])]
print(f"\nMissing from summaries but have PA records: {len(missing_with_pa)}")

Earliest missing: 1953-05-02 00:00:00+00:00
Latest missing: 2020-01-20 00:00:00+00:00

Missing by incident type:
incidentType
Flood                637
Severe Storm         205
Tornado              121
Hurricane             98
Snowstorm             34
Typhoon               34
Fire                  25
Earthquake            17
Freezing              13
Coastal Storm         10
Drought               10
Severe Ice Storm       9
Other                  8
Fishing Losses         5
Volcanic Eruption      4
Toxic Substances       3
Dam/Levee Break        2
Human Cause            2
Mud/Landslide          1
Biological             1
Name: count, dtype: int64

Missing from summaries but have PA records: 1


## Joined Dataset

In [41]:
df = declarations_df_collapsed.merge(
    disaster_summaries_df, on='disasterNumber', how='inner'
).merge(
    public_assistance_agg_df, on='disasterNumber', how='left'
).drop(columns=['paLoadDate', 'iaLoadDate', 'incidentEndDate'])

df.shape

(3940, 36)

In [43]:
df = df.fillna(0)

In [44]:
df.isna().sum()

disasterNumber                0
declarationDate               0
incidentType                  0
declarationType               0
state                         0
fyDeclared                    0
incidentBeginDate             0
designatedArea_count          0
ihProgramDeclared             0
iaProgramDeclared             0
paProgramDeclared             0
hmProgramDeclared             0
incident_duration_days        0
is_ongoing                    0
totalAmountIhpApproved        0
totalAmountHaApproved         0
totalAmountOnaApproved        0
totalObligatedAmountPa        0
totalObligatedAmountCatAb     0
totalObligatedAmountCatC2g    0
totalObligatedAmountHmgp      0
total_disaster_cost           0
total_obligated               0
federal_obligated             0
avg_project_cost              0
applicant_count               0
project_count                 0
large_project_count           0
small_project_count           0
has_debris                    0
has_emergency                 0
has_road

In [45]:
CLEANED_DATA_DIR = Path("__file__").resolve().parent.parent / "data" / "cleaned"

In [46]:
df.to_csv(CLEANED_DATA_DIR / "cleaned_data.csv", index=False)